In [1]:
import hashlib
import numpy as np
import time
import pandas as pd
import pickle

class CuckooFilter:
    """
    Cuckoo Filter for fast O(1) threat signature lookup.
    KEY ADVANTAGE over Bloom Filter: supports DELETE operation.
    This allows dynamic threat intel updates at edge without rebuilding.
    """
    def __init__(self, capacity=20000, bucket_size=4, max_kicks=500):
        self.capacity    = capacity
        self.bucket_size = bucket_size
        self.max_kicks   = max_kicks
        self.buckets     = [[None]*bucket_size for _ in range(capacity)]
        self.size        = 0

    def _fingerprint(self, item):
        return hashlib.md5(str(item).encode()).hexdigest()[:8]

    def _hash1(self, item):
        return int(hashlib.sha256(str(item).encode()).hexdigest(), 16) % self.capacity

    def _hash2(self, fp, h1):
        return (h1 ^ int(hashlib.sha256(fp.encode()).hexdigest(), 16)) % self.capacity

    def insert(self, item):
        fp = self._fingerprint(item)
        i1 = self._hash1(item)
        i2 = self._hash2(fp, i1)
        for i in [i1, i2]:
            if None in self.buckets[i]:
                self.buckets[i][self.buckets[i].index(None)] = fp
                self.size += 1
                return True
        i = i1
        for _ in range(self.max_kicks):
            slot = np.random.randint(self.bucket_size)
            fp, self.buckets[i][slot] = self.buckets[i][slot], fp
            i = self._hash2(fp, i)
            if None in self.buckets[i]:
                self.buckets[i][self.buckets[i].index(None)] = fp
                self.size += 1
                return True
        return False

    def lookup(self, item):
        fp = self._fingerprint(item)
        i1 = self._hash1(item)
        i2 = self._hash2(fp, i1)
        return fp in self.buckets[i1] or fp in self.buckets[i2]

    def delete(self, item):
        """Bloom Filter CANNOT do this — our key innovation"""
        fp = self._fingerprint(item)
        i1 = self._hash1(item)
        i2 = self._hash2(fp, i1)
        for i in [i1, i2]:
            if fp in self.buckets[i]:
                self.buckets[i][self.buckets[i].index(fp)] = None
                self.size -= 1
                return True
        return False

print("✓ CuckooFilter class defined")

✓ CuckooFilter class defined


In [6]:
df = pd.read_csv('../data/my_iot_dataset.csv')

def make_signature(row):
    return f"{round(row['Rate'], 1)}_{row['Protocol Type']}_{row['ICMP']}"

attack_df = df[df['category'] != 'Benign']
benign_df = df[df['category'] == 'Benign']

# Get signatures from each group
attack_sigs = set(attack_df.apply(make_signature, axis=1).unique())
benign_sigs = set(benign_df.apply(make_signature, axis=1).unique())

# ONLY keep signatures that NEVER appear in benign traffic
exclusive_attack_sigs = attack_sigs - benign_sigs

print(f"Total attack signatures  : {len(attack_sigs)}")
print(f"Total benign signatures  : {len(benign_sigs)}")
print(f"Overlapping signatures   : {len(attack_sigs & benign_sigs)}")
print(f"Exclusive attack sigs    : {len(exclusive_attack_sigs)}")

Total attack signatures  : 20911
Total benign signatures  : 6467
Overlapping signatures   : 2198
Exclusive attack sigs    : 18713


In [7]:
df = pd.read_csv('../data/my_iot_dataset.csv')

def make_signature(row):
    return f"{round(row['Rate'], 1)}_{row['Protocol Type']}_{row['ICMP']}"

attack_df = df[df['category'] != 'Benign']
benign_df = df[df['category'] == 'Benign']

# Get signatures from each group
attack_sigs = set(attack_df.apply(make_signature, axis=1).unique())
benign_sigs = set(benign_df.apply(make_signature, axis=1).unique())

# ONLY keep signatures that NEVER appear in benign traffic
exclusive_attack_sigs = attack_sigs - benign_sigs

print(f"Total attack signatures  : {len(attack_sigs)}")
print(f"Total benign signatures  : {len(benign_sigs)}")
print(f"Overlapping signatures   : {len(attack_sigs & benign_sigs)}")
print(f"Exclusive attack sigs    : {len(exclusive_attack_sigs)}")

Total attack signatures  : 20911
Total benign signatures  : 6467
Overlapping signatures   : 2198
Exclusive attack sigs    : 18713


In [4]:
# Show delete works — Bloom Filter cannot do this
test_sig = unique_sigs[0]

print(f"Signature: '{test_sig}'")
print(f"Lookup before delete : {cf.lookup(test_sig)}")   # True
cf.delete(test_sig)
print(f"Lookup after delete  : {cf.lookup(test_sig)}")   # False
cf.insert(test_sig)
print(f"Lookup after re-insert: {cf.lookup(test_sig)}")  # True
print("\n✓ Dynamic insert/delete works — Bloom Filter cannot do this!")

Signature: '26.2_6.0_0.0'
Lookup before delete : True
Lookup after delete  : False
Lookup after re-insert: True

✓ Dynamic insert/delete works — Bloom Filter cannot do this!


In [5]:
pickle.dump(cf, open('../models/cuckoo_filter.pkl', 'wb'))
pickle.dump(make_signature, open('../models/signature_fn.pkl', 'wb'))
print("✓ Cuckoo filter saved to models/cuckoo_filter.pkl")

✓ Cuckoo filter saved to models/cuckoo_filter.pkl


In [8]:
df = pd.read_csv('../data/my_iot_dataset.csv')

def make_signature(row):
    return f"{round(row['Rate'], 1)}_{row['Protocol Type']}_{row['ICMP']}"

attack_df = df[df['category'] != 'Benign']
benign_df = df[df['category'] == 'Benign']

# Get signatures from each group
attack_sigs = set(attack_df.apply(make_signature, axis=1).unique())
benign_sigs = set(benign_df.apply(make_signature, axis=1).unique())

# ONLY keep signatures that NEVER appear in benign traffic
exclusive_attack_sigs = attack_sigs - benign_sigs

print(f"Total attack signatures  : {len(attack_sigs)}")
print(f"Total benign signatures  : {len(benign_sigs)}")
print(f"Overlapping signatures   : {len(attack_sigs & benign_sigs)}")
print(f"Exclusive attack sigs    : {len(exclusive_attack_sigs)}")

Total attack signatures  : 20911
Total benign signatures  : 6467
Overlapping signatures   : 2198
Exclusive attack sigs    : 18713


In [9]:
cf_clean = CuckooFilter(capacity=50000)

inserted = 0
for sig in exclusive_attack_sigs:
    if cf_clean.insert(sig):
        inserted += 1

print(f"✓ Clean filter built: {inserted} exclusive signatures")

# Save — overwrite old filter
pickle.dump(cf_clean, open('../models/cuckoo_filter.pkl', 'wb'))
print("✓ Clean Cuckoo Filter saved — old one overwritten")

✓ Clean filter built: 18713 exclusive signatures
✓ Clean Cuckoo Filter saved — old one overwritten


In [10]:
# Test on benign traffic — should get 0 hits now
benign_test = benign_df.sample(500, random_state=42)
benign_sigs_test = benign_test.apply(make_signature, axis=1).values

false_hits = sum(1 for s in benign_sigs_test if cf_clean.lookup(s))
total      = len(benign_sigs_test)

print(f"Benign packets tested    : {total}")
print(f"Wrongly flagged (FP)     : {false_hits}")
print(f"False Positive Rate      : {false_hits/total*100:.1f}%")
print(f"\nTarget: 0% — {'✓ PASSED' if false_hits == 0 else '✗ Still has overlap'}")

Benign packets tested    : 500
Wrongly flagged (FP)     : 0
False Positive Rate      : 0.0%

Target: 0% — ✓ PASSED
